In [10]:
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from transformers import TrainingArguments, Trainer

In [11]:
df = pd.read_csv("../symptoms_dataset.csv")

df = df.dropna()
df["text"] = df["text"].str.strip()
df["disease"] = df["disease"].str.strip()

print(df.shape)
print(df["disease"].nunique())

(375, 2)
15


In [12]:
le = LabelEncoder()
df["label"] = le.fit_transform(df["disease"])

num_labels = len(le.classes_)

print("Classes:", num_labels)

Classes: 15


In [13]:
X_train, X_val, y_train, y_val = train_test_split(
    df["text"].tolist(),
    df["label"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

In [14]:
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

train_enc = tokenizer(X_train, truncation=True, padding=True, max_length=128)
val_enc = tokenizer(X_val, truncation=True, padding=True, max_length=128)

In [15]:
class PlantDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_ds = PlantDataset(train_enc, y_train)
val_ds = PlantDataset(val_enc, y_val)

In [16]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2612.04it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
args = TrainingArguments(
    output_dir="./distilbert_model",
    eval_strategy="epoch",   # IMPORTANT: correct for your version
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    logging_steps=10
)

In [18]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds
)

In [19]:
trainer.train()

c:\Users\FATIMA\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,2.698255,2.604157
2,2.545888,2.234929
3,2.183330,1.887236
4,1.888318,1.678939
5,1.737734,1.604451


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.08it/s]
c:\Users\FATIMA\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.18s/it]
c:\Users\FATIMA\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.20s/it]
c:\Users\FATIMA\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100

TrainOutput(global_step=95, training_loss=2.192771339416504, metrics={'train_runtime': 139.6798, 'train_samples_per_second': 10.739, 'train_steps_per_second': 0.68, 'total_flos': 7375383045000.0, 'train_loss': 2.192771339416504, 'epoch': 5.0})

In [20]:
model.save_pretrained("distilbert_plant_model")
tokenizer.save_pretrained("distilbert_plant_model")

import joblib
joblib.dump(le, "label_encoder.pkl")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.33it/s]


['label_encoder.pkl']

In [21]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)

    pred = torch.argmax(outputs.logits, dim=1).item()
    return le.inverse_transform([pred])[0]

print(predict("brown spots spreading on tomato leaves"))

Tomato_Leaf_Mold
